In [ ]:
# Shared lighting robustness evaluation for both persisted models:
# 1) classic SVM pipeline (GLCM + DWT)
# 2) transfer-learning CNN model
import json
from pathlib import Path

import joblib
import numpy as np
import tensorflow as tf
from sklearn.pipeline import Pipeline

from steelblast_classic_features import FeatureExtractionConfig, load_split
from steelblast_lighting_analysis import (
    build_svm_predict_fn,
    build_transfer_predict_fn,
    run_robustness_analysis,
)


# Shared dataset setup
# Uses the same test split for both models so robustness outcomes are directly comparable.
dataset_dir = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
test_paths, y_test = load_split(dataset_dir, "test")

# Configurable thresholds for robustness claims
CLAIM = "Model is robust to non-saturating lighting variation."
CRITERION = "For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06."
MIN_CLAIM_ACCURACY = 0.90
MAX_CLAIM_FLIP_RATE = 0.06
NOTE = "Severe brightening is reported as a stress test because clipping can destroy texture information."

In [ ]:


# Analyze SVM model
svm_model_path = Path("steelblast_svm_glcm_dwt.joblib")
if not svm_model_path.exists():
    raise FileNotFoundError(f"Saved SVM model not found: {svm_model_path.resolve()}")

svm_feature_config = FeatureExtractionConfig(
    image_size=256,
    illumination_normalization="clahe",
    clahe_clip_limit=0.01,
    glcm_levels=32,
    glcm_distances=(1, 2, 4, 8),
    glcm_angles=(0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    glcm_properties=("contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"),
    dwt_wavelet="db2",
    dwt_level=3,
)

svm_model: Pipeline = joblib.load(svm_model_path)
svm_predict_for_perturbation = build_svm_predict_fn(
    image_paths=test_paths,
    feature_config=svm_feature_config,
    fitted_model=svm_model,
)
svm_lighting_robustness = run_robustness_analysis(
    model_name="SVM (GLCM + DWT)",
    y_test=y_test,
    predict_for_perturbation=svm_predict_for_perturbation,
    claim=CLAIM,
    criterion=CRITERION,
    min_accuracy_threshold=MIN_CLAIM_ACCURACY,
    max_flip_rate_threshold=MAX_CLAIM_FLIP_RATE,
    note=NOTE,
)

svm_metrics_path = Path("bias_analysis_metrics_svm_glcm_dwt.json")
svm_metrics = {
    "dataset_dir": str(dataset_dir),
    "model_key": "svm_glcm_dwt",
    "model_path": str(svm_model_path),
    "lighting_robustness": svm_lighting_robustness,
}
svm_metrics_path.write_text(json.dumps(svm_metrics, indent=2))
print(f"\nSaved SVM lighting robustness results to: {svm_metrics_path.resolve()}")



In [ ]:
# Analyze transfer-learning model
transfer_model_path = Path("steelblast_resnet50.h5")
if not transfer_model_path.exists():
    raise FileNotFoundError(f"Saved transfer model not found: {transfer_model_path.resolve()}")

transfer_model = tf.keras.models.load_model(transfer_model_path, compile=False)
transfer_predict_for_perturbation = build_transfer_predict_fn(
    image_paths=test_paths,
    model=transfer_model,
    image_size=(224, 224),
    preprocess_input_fn=tf.keras.applications.resnet50.preprocess_input,
    threshold=0.5,
)
transfer_lighting_robustness = run_robustness_analysis(
    model_name="Transfer Learning (ResNet50)",
    y_test=y_test,
    predict_for_perturbation=transfer_predict_for_perturbation,
    claim=CLAIM,
    criterion=CRITERION,
    min_accuracy_threshold=MIN_CLAIM_ACCURACY,
    max_flip_rate_threshold=MAX_CLAIM_FLIP_RATE,
    note=NOTE,
)


transfer_metrics_path = Path("bias_analysis_metrics_transfer_resnet50.json")
transfer_metrics = {
    "dataset_dir": str(dataset_dir),
    "model_key": "transfer_resnet50",
    "model_path": str(transfer_model_path),
    "lighting_robustness": transfer_lighting_robustness,
}
transfer_metrics_path.write_text(json.dumps(transfer_metrics, indent=2))
print(f"Saved transfer lighting robustness results to: {transfer_metrics_path.resolve()}")